# Context Window Awareness & Cost Management

In this notebook you will learn how to see how full an agent's context window is, how to read what each call cost, and how to put guardrails around a run so it cannot get out of hand.

You will work with a small, read-only agent that answers questions about a seed library inventory. Every agent call in this notebook only uses the `Read` tool, so the files on disk never change. At the end you will prove that with file hashes.

| Cell | What it does |
| --- | --- |
| 1 | Installs the Claude Agent SDK |
| 2 | Loads your API key from Colab Secrets |
| 3 | Sets the model name once for the whole notebook |
| 4 | Declares the price constants used for cost estimates |
| 5 | Creates the demo files and records their SHA-256 hashes |
| 6 | Verifies the files before any agent runs |
| 7 | Shows the context window growing across a three-turn conversation |
| 8 | Reads the cost and usage of one call |
| 9 | Estimates that cost yourself and compares it with the SDK |
| 10 | Shows how cost behaves across a resumed session |
| 11 | Compacts the conversation and measures the effect |
| 12 | Stops runs with `max_turns` and `max_budget_usd` |
| 13 | Compares the default `effort` level with `effort="low"` |
| 14 | Confirms the files are unchanged |

Agent output varies from run to run. That is normal for real agents. In every cell, look for the pattern described in the text, not for identical numbers.

## Cell 1: Install the Claude Agent SDK

This cell installs `claude-agent-sdk`, the Python package that lets you run Claude as an agent from your own code. The package also bundles the command line tool that actually runs the agent loop, so this is the only install the notebook needs.

The version is pinned so that every example behaves the way it does in the video. If the package is already present in your Colab session, pip checks that the pinned version is installed and finishes quickly. The `-q` flag keeps the output quiet.

Run the cell and wait for it to finish before moving on.

In [1]:
# Cell 1: install the SDK
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.167 -q

## Cell 2: Load your API key from Colab Secrets

The agent needs an Anthropic API key. You create the key in the Claude Console at https://platform.claude.com/ under **API Keys**. Never paste a key directly into a notebook cell, because notebooks get shared.

Store the key as a Colab Secret instead:

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key as the value.
5. Switch **Notebook access** on for this notebook.

The cell below reads the secret and writes it into the `ANTHROPIC_API_KEY` environment variable, which is where the SDK looks for it.

If you are running this notebook locally instead of in Colab, skip the Colab import and set the environment variable in your terminal before you start Jupyter: `export ANTHROPIC_API_KEY="your-key"`.

In [2]:
# Cell 2: API key from Colab Secrets
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model once

Every agent call in this notebook reads the model name from a single variable, `MODEL_NAME`. If you change the value in this cell and rerun the notebook, the model changes everywhere.

This notebook uses Claude Haiku 5.5 because it is fast and inexpensive, which suits a lecture where you will run many calls. To see which models are currently available, use the models page linked in the comment.

In [3]:
# Cell 3: the model used across this notebook
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-5-5"

## Cell 4: Price constants for cost estimates

A request’s prompt is split into three non-overlapping parts

| Field in `usage` | What falls into it | How it is billed |
| --- | --- | --- |
| `input_tokens` | Prompt tokens that were neither written to nor read from the cache | Ordinary input rate |
| `cache_creation_input_tokens` | Prompt tokens stored in the cache on this call (a **cache write**) | Cache write rate |
| `cache_read_input_tokens` | Prompt tokens served from the cache (a **cache read**) | Cache read rate |
| `output_tokens` | Tokens the model writes back | Output rate |

Each prompt token is billed in **exactly one** of the first three rows, so nothing is counted twice. The full prompt size is `input_tokens` + `cache_creation_input_tokens` + `cache_read_input_tokens`.

Later in this notebook you will estimate what a call cost from its token counts. To do that you need prices, and prices are measured per **million tokens** (MTok). This cell keeps all of them in one place so that you only have one cell to update when prices change.

The rates below are the Claude Haiku 5.5 rates for prompts of up to 100,000 tokens, taken from the pricing page at https://platform.claude.com/docs/en/about-claude/pricing on 2026-10-19. The page does not show a last-updated date, and **rates change**, so check the page for current figures before you rely on these numbers.

| Constant | Meaning | USD per million tokens |
| --- | --- | --- |
| `PRICE_INPUT` | Ordinary input tokens | 0.10 |
| `PRICE_OUTPUT` | Output tokens | 0.50 |
| `PRICE_CACHE_WRITE_5M` | Input tokens written to the 5-minute prompt cache | 0.125 |
| `PRICE_CACHE_READ` | Input tokens read back from the prompt cache | 0.01 |

**The long-prompt tier.** Claude Haiku 5.5 is priced by prompt length. A request whose prompt is over 100,000 tokens pays a higher tier: 0.50 for input, 2.50 for output, 0.625 for a 5-minute cache write and 0.05 for a cache read. The higher tier applies to the **whole request**, not just to the tokens beyond the threshold. The pricing page counts cache reads and cache writes as part of the prompt length.

Every demo in this notebook stays far below 100,000 tokens, so this notebook uses only the lower tier and contains no tier-switching code. Treat the estimates you compute as approximate.

In [4]:
# Cell 4: price constants (USD per million tokens, Claude Haiku 5.5)
# Source: https://platform.claude.com/docs/en/about-claude/pricing
# Rates change over time. Check the page for current figures.
# These are the rates for prompts up to 100,000 tokens.
PRICE_INPUT = 0.10
PRICE_OUTPUT = 0.50
PRICE_CACHE_WRITE_5M = 0.125
PRICE_CACHE_READ = 0.01

## Cell 5: Create the demo files and record their hashes

The agent needs something to read. This cell creates two files under `/content/seedlibrary_demo/`:

- `seed_inventory.txt` holds 600 inventory lines such as `SL-0001 | amber marigold | 40 packets | shelf A1`. The data is generated by a plain loop with no randomness, so everyone gets identical files. One seed, on line `SL-0417`, holds far more packets than any other. That gives every demo a clear, checkable answer.
- `notes.txt` holds four short lines. It is deliberately tiny, so you can compare a small read with a large one.

The inventory is large on purpose. Reading it adds thousands of tokens to the conversation in a single step, which is exactly the behavior this lecture is about.

Finally the cell computes a **SHA-256 hash** of each file. A hash is a fingerprint of the file's exact contents. Every agent call in this notebook is read-only, so in Cell 14 the hashes must still match.

In [5]:
# Cell 5: create the demo files and record their hashes
import hashlib
from pathlib import Path

DEMO_DIR = Path("/content/seedlibrary_demo")
DEMO_DIR.mkdir(parents=True, exist_ok=True)
INVENTORY = DEMO_DIR / "seed_inventory.txt"
NOTES = DEMO_DIR / "notes.txt"

ADJECTIVES = [
    "amber", "bright", "crimson", "dusty", "early",
    "frosted", "golden", "hardy", "ivory", "jade",
    "late", "mellow", "nimble", "olive", "pale",
    "quiet", "rustic", "silver", "tawny", "wild",
]
PLANTS = [
    "marigold", "zinnia", "basil", "dill", "cosmos",
    "lupine", "sorrel", "fennel", "borage", "yarrow",
    "poppy", "sunflower", "chard", "kale", "radish",
    "beet", "lettuce", "spinach", "pea", "bean",
    "squash", "cucumber", "pepper", "tomato", "carrot",
    "turnip", "leek", "onion", "parsley", "thyme",
]

inventory_lines = []
for i in range(600):
    seed_id = f"SL-{i + 1:04d}"
    seed_name = f"{ADJECTIVES[i % 20]} {PLANTS[i // 20]}"
    packets = 5 + (i * 7) % 90
    if i == 416:
        packets = 480
    shelf = f"{'ABCDEFGH'[i % 8]}{(i * 3) % 9 + 1}"
    inventory_lines.append(f"{seed_id} | {seed_name} | {packets} packets | shelf {shelf}")

INVENTORY.write_text("\n".join(inventory_lines) + "\n")

NOTES.write_text(
    "Seed library notes\n"
    "Shelves A to H hold the main stock.\n"
    "Counts were last checked in spring.\n"
    "Ask the front desk before borrowing more than ten packets.\n"
)


def sha256_of(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


BASELINE_HASHES = {
    "seed_inventory.txt": sha256_of(INVENTORY),
    "notes.txt": sha256_of(NOTES),
}

for name, digest in BASELINE_HASHES.items():
    print(f"{name}: {digest}")

seed_inventory.txt: 778dca87088ca5b285c1b7966db24d521185559eefcc2495c9adc98521a802fe
notes.txt: 2cd1f97592bf2686ff9c1cdb23528c3536223ced524f267108750da481d37a69


## Cell 6: Verify the files before any agent runs

Before an agent touches anything, confirm that the files are what you think they are. This cell prints the directory tree with file sizes, the line count of each file, the first few lines of the inventory, and the contents of the notes.

It also computes the correct answer to the question the agent will be asked, using ordinary Python. That gives you **ground truth** to compare against: if the agent says a different seed holds the most packets, you will know it is wrong.

Look for three things in the output: 600 lines in the inventory, 4 lines in the notes, and `SL-0417` as the line with the most packets.

In [6]:
# Cell 6: verify the files and compute the ground-truth answer
print(DEMO_DIR)
for path in sorted(DEMO_DIR.rglob("*")):
    size = path.stat().st_size
    line_count = len(path.read_text().splitlines())
    print(f"  {path.name}  ({size:,} bytes, {line_count} lines)")

print("\nFirst 5 lines of seed_inventory.txt:")
for line in inventory_lines[:5]:
    print(" ", line)

print("\nnotes.txt:")
print(NOTES.read_text())


def packets_on(line):
    return int(line.split("|")[2].split()[0])


top_line = max(inventory_lines, key=packets_on)
print("Ground truth, most packets:")
print(" ", top_line)

/content/seedlibrary_demo
  notes.txt  (150 bytes, 4 lines)
  seed_inventory.txt  (28,186 bytes, 600 lines)

First 5 lines of seed_inventory.txt:
  SL-0001 | amber marigold | 5 packets | shelf A1
  SL-0002 | bright marigold | 12 packets | shelf B4
  SL-0003 | crimson marigold | 19 packets | shelf C7
  SL-0004 | dusty marigold | 26 packets | shelf D1
  SL-0005 | early marigold | 33 packets | shelf E4

notes.txt:
Seed library notes
Shelves A to H hold the main stock.
Counts were last checked in spring.
Ask the front desk before borrowing more than ten packets.

Ground truth, most packets:
  SL-0417 | rustic squash | 480 packets | shelf A7


## Cell 7: The context window grows with the conversation

An agent's **context window** is everything Claude can see while it works: the system prompt, the tool definitions, the conversation history, and every tool output. It does **not** reset between turns. Everything piles up, and reading a large file can add thousands of tokens in one step.

To see this happen, this cell uses `ClaudeSDKClient`, which keeps one conversation going across several prompts. It asks three questions:

1. A general one-sentence question. No tool is needed, so the context stays small.
2. A request to read the inventory. The whole file lands in the context.
3. A question about the first thing you asked. Claude can answer only because the conversation remembers.

The method `await client.get_context_usage()` returns a breakdown of the context window. This cell calls it before the first turn and after each turn. The fields it prints are:

| Field | Meaning |
| --- | --- |
| `totalTokens` | Tokens currently in the context window |
| `maxTokens` | The effective limit that usage is measured against |
| `percentage` | `totalTokens` as a percentage of the window |
| `categories` | The breakdown by source, such as system prompt, tools and messages |

Read `totalTokens` first and the percentage second. Claude Haiku 5.5 has a 1M-token window, so the percentage stays small and moves in tiny steps. The token count is the number that tells the story.

`get_context_usage()` is documented in the SDK source code rather than the main documentation pages, and the exact categories it returns can differ between versions. The cell skips any "free space" or buffer entry so the printed list shows only what is using context.

Watch for this: `totalTokens` should jump after turn 2, because the file was read, and then barely move after turn 3. Turn 3 adds only a short question and a short answer. It is the payoff of "the conversation remembers".

Notice that `allowed_tools` contains only `Read`. The agent needs nothing else, and listing only what a demo needs is a good habit.

In [7]:
# Cell 7: watch the context window grow across three turns
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, ResultMessage

SKIP_CATEGORIES = ("free space", "buffer")


def show_context(label, usage):
    print(f"--- {label} ---")
    print(f"  totalTokens: {usage['totalTokens']:,}")
    print(f"  maxTokens:   {usage['maxTokens']:,}")
    print(f"  percentage:  {usage['percentage']:.2f}%")
    in_use = [
        category for category in usage["categories"]
        if not any(skip in category["name"].lower() for skip in SKIP_CATEGORIES)
    ]
    in_use.sort(key=lambda category: category["tokens"], reverse=True)
    print("  largest categories:")
    for category in in_use[:4]:
        print(f"    {category['name']}: {category['tokens']:,}")


TURNS = [
    "In one sentence, what is a seed library?",
    f"Read {INVENTORY} and tell me how many lines it has and which seed has the most packets.",
    "What was the first question I asked you?",
]

options = ClaudeAgentOptions(
    allowed_tools=["Read"],
    model=MODEL_NAME,
    cwd="/content",
)

async with ClaudeSDKClient(options=options) as client:
    show_context("Before turn 1", await client.get_context_usage())

    for number, prompt in enumerate(TURNS, start=1):
        print(f"\nTurn {number}: {prompt}")
        await client.query(prompt)
        async for message in client.receive_response():
            if isinstance(message, ResultMessage) and message.subtype == "success":
                print(f"Answer: {message.result}")
        show_context(f"After turn {number}", await client.get_context_usage())

--- Before turn 1 ---
  totalTokens: 10,643
  maxTokens:   1,000,000
  percentage:  1.00%
  largest categories:
    System tools (deferred): 12,216
    System tools: 8,525
    Skills: 1,966
    System prompt: 142

Turn 1: In one sentence, what is a seed library?
Answer: A seed library is a collection of stored seeds, usually from plants, kept by a person, community, or institution so that plant varieties can be preserved, shared, and grown again in future seasons.
--- After turn 1 ---
  totalTokens: 14,192
  maxTokens:   1,000,000
  percentage:  1.00%
  largest categories:
    System tools (deferred): 12,216
    System tools: 8,590
    Messages: 3,494
    Skills: 1,966

Turn 2: Read /content/seedlibrary_demo/seed_inventory.txt and tell me how many lines it has and which seed has the most packets.
Answer: The file has **600 lines**, one per seed entry (SL-0001 through SL-0600).

The seed with the most packets is **SL-0417, rustic squash, with 480 packets** and shelf A7. The next highest

## Cell 8: Read what one call cost

Now switch from the context window to the bill. The SDK function `query()` does not return a value. It returns an **async stream of messages** while the agent loop runs on its own. The last message in that stream is a `ResultMessage`, and that is where the cost information lives.

This cell runs one `query()` call, keeps the `ResultMessage`, and prints these fields:

| Field | Meaning |
| --- | --- |
| `subtype` | How the run ended. `success` means it finished normally |
| `num_turns` | How many tool-use round trips the loop took |
| `total_cost_usd` | The SDK's estimate of what the call cost, in US dollars |
| `usage` | A dict of token counts for the main agent loop |
| `model_usage` | Per-model token counts and a per-model cost |

Three things to remember about these fields:

- In Python `total_cost_usd` is optional, so the cell checks for `None` before printing it.
- `total_cost_usd` is a **client-side estimate** computed from a price table inside the SDK. It is not your bill. The authoritative figures are on the Usage page of the Claude Console at https://platform.claude.com/usage.
- Output tokens must be read from the `ResultMessage`. The output count on individual assistant messages is only a placeholder.

The `usage` dict includes `input_tokens`, `output_tokens`, `cache_creation_input_tokens` and `cache_read_input_tokens`. The SDK uses prompt caching automatically, so do not be surprised if cache tokens make up most of the input.

The cell also prints the per-model entry from `model_usage`: its `costUSD`, its `contextWindow`, and a `canonicalModel` name if the SDK provides one. This shows which model name the SDK used to look up a price.

Run the cell and look for the shape of the numbers: a small `num_turns`, a `total_cost_usd` that is a tiny fraction of a cent or a few cents, and a `usage` dict whose largest counts are on the input side.

In [8]:
# Cell 8: read the cost and usage of one call
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage
import json

READ_PROMPT = (
    f"Read {INVENTORY} and tell me which seed has the most packets. "
    "Answer in one sentence."
)

result_8 = None
async for message in query(
    prompt=READ_PROMPT,
    options=ClaudeAgentOptions(
        allowed_tools=["Read"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage):
        result_8 = message

print("subtype:   ", result_8.subtype)
print("num_turns: ", result_8.num_turns)
if result_8.total_cost_usd is None:
    print("total_cost_usd: not reported")
else:
    print(f"total_cost_usd: ${result_8.total_cost_usd:.6f}")

print("\nusage:")
print(json.dumps(result_8.usage, indent=2))

print("\nmodel_usage:")
for model_name, entry in (result_8.model_usage or {}).items():
    print(f"  model key:      {model_name}")
    print(f"  canonicalModel: {entry.get('canonicalModel')}")
    print(f"  costUSD:        ${entry['costUSD']:.6f}")
    print(f"  contextWindow:  {entry['contextWindow']:,}")

subtype:    success
num_turns:  2
total_cost_usd: $0.003012

usage:
{
  "input_tokens": 4,
  "cache_creation_input_tokens": 19930,
  "cache_read_input_tokens": 24956,
  "output_tokens": 281,
  "output_tokens_details": {
    "thinking_tokens": 146
  },
  "server_tool_use": {
    "web_search_requests": 0,
    "web_fetch_requests": 0
  },
  "service_tier": "standard",
  "cache_creation": {
    "ephemeral_1h_input_tokens": 0,
    "ephemeral_5m_input_tokens": 19930
  },
  "inference_geo": "global",
  "iterations": [
    {
      "input_tokens": 2,
      "output_tokens": 212,
      "cache_read_input_tokens": 14223,
      "cache_creation_input_tokens": 16440,
      "cache_creation": {
        "ephemeral_5m_input_tokens": 16440,
        "ephemeral_1h_input_tokens": 0
      },
      "type": "message"
    }
  ],
  "speed": "standard",
  "fallback_credit": null
}

model_usage:
  model key:      claude-haiku-5-5
  canonicalModel: claude-haiku-5-5
  costUSD:        $0.003012
  contextWindow:  1,000,

## Cell 9: Estimate the cost yourself

You do not have to take `total_cost_usd` on trust. The `usage` dict gives you the token counts, and Cell 4 gave you the rates. The arithmetic is:

> cost = (input x input rate + output x output rate + cache writes x cache write rate + cache reads x cache read rate) / 1,000,000

The division is there because the rates are per million tokens. This cell applies that formula to the `ResultMessage` from Cell 8, prints each line of the calculation, and then prints your estimate next to the SDK's `total_cost_usd` together with the difference.

Expect the two figures to be **close but not necessarily identical to the last digit**. That is a feature of estimates, not a bug. Some reasons they can differ:

- Adaptive thinking is on by default, and it is not guaranteed that thinking tokens appear in the `output_tokens` count. If they are missing, your estimate can come out a little low.
- The SDK prices each request inside the loop separately, while you price the totals.
- The SDK uses its own built-in price table, which may be a version behind the pricing page.
- This cell assumes the 5-minute cache write rate and the lower prompt-length tier.

If the two numbers are within a small fraction of each other, your mental model of the cost is sound. If they are wildly different, check that the model name was recognized in the `model_usage` output of Cell 8.

Remember that both numbers are estimates. The Usage page in the Claude Console remains the source of truth for what you are billed.

In [9]:
# Cell 9: estimate the cost from usage counts and the Cell 4 rates
usage = result_8.usage or {}
input_tokens = usage.get("input_tokens", 0)
output_tokens = usage.get("output_tokens", 0)
cache_write_tokens = usage.get("cache_creation_input_tokens", 0)
cache_read_tokens = usage.get("cache_read_input_tokens", 0)

parts = [
    ("input", input_tokens, PRICE_INPUT),
    ("output", output_tokens, PRICE_OUTPUT),
    ("cache write", cache_write_tokens, PRICE_CACHE_WRITE_5M),
    ("cache read", cache_read_tokens, PRICE_CACHE_READ),
]

my_estimate = 0.0
for label, tokens, rate in parts:
    dollars = tokens * rate / 1_000_000
    my_estimate += dollars
    print(f"{label:<12} {tokens:>9,} tokens x ${rate} per MTok = ${dollars:.6f}")

print(f"\nYour estimate:   ${my_estimate:.6f}")
if result_8.total_cost_usd is None:
    print("SDK total_cost_usd was not reported, so there is nothing to compare.")
else:
    difference = my_estimate - result_8.total_cost_usd
    print(f"SDK total_cost_usd: ${result_8.total_cost_usd:.6f}")
    print(f"Difference:      ${difference:+.6f}")

input                4 tokens x $0.1 per MTok = $0.000000
output             281 tokens x $0.5 per MTok = $0.000140
cache write     19,930 tokens x $0.125 per MTok = $0.002491
cache read      24,956 tokens x $0.01 per MTok = $0.000250

Your estimate:   $0.002882
SDK total_cost_usd: $0.003012
Difference:      $-0.000130


## Cell 10: Cost across a resumed session

Sometimes you continue an earlier conversation in a new call by passing `resume=session_id` in the options. The session ID comes from the `session_id` field of the earlier `ResultMessage`.

Here is the part that trips people up: a **resumed call's result already includes the session's earlier spend**. Its `total_cost_usd` is the running total for the whole session, not just the new call. So to get the session total you read the **latest** result. If you add the results together, you count the earlier spend twice.

This cell runs two calls:

- **Call A** is a fresh `query()` that reads the inventory.
- **Call B** resumes call A's session and asks a follow-up that needs no tool.

It then prints both costs, the correct session total (the latest result), and the naive A plus B total, so you can see the double count.

The cell defines a small helper called `run_once`. It runs one `query()` call with read-only options, returns the final `ResultMessage`, and also catches any exception the call raises. You will reuse it in Cells 12 and 13. Any extra option you pass, such as `resume=...`, is forwarded to `ClaudeAgentOptions`.

Expect `A + B` to be noticeably larger than B alone, and expect B to be at least as large as A. If B is **not** larger than A, resumed totals were not carried forward in your setup, and you should tell your instructor, because the lesson depends on it.

Either way, this combined figure is still a client-side estimate.

In [10]:
# Cell 10: cost across a resumed session
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage


async def run_once(prompt, **extra_options):
    options = ClaudeAgentOptions(
        allowed_tools=["Read"],
        model=MODEL_NAME,
        cwd="/content",
        **extra_options,
    )
    final = None
    error = None
    try:
        async for message in query(prompt=prompt, options=options):
            if isinstance(message, ResultMessage):
                final = message
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
    return final, error


def money(value):
    return "not reported" if value is None else f"${value:.6f}"


call_a, error_a = await run_once(READ_PROMPT)
call_b, error_b = await run_once(
    "How many packets does that seed hold? Answer in one short sentence.",
    resume=call_a.session_id,
)

print(f"Call A total_cost_usd: {money(call_a.total_cost_usd)}")
print(f"Call B total_cost_usd: {money(call_b.total_cost_usd)}")

if call_a.total_cost_usd is not None and call_b.total_cost_usd is not None:
    print(f"\nCorrect session total (latest result, B): {money(call_b.total_cost_usd)}")
    naive_total = call_a.total_cost_usd + call_b.total_cost_usd
    print(f"Naive A + B (double counts A):            {money(naive_total)}")

Call A total_cost_usd: $0.000653
Call B total_cost_usd: $0.000987

Correct session total (latest result, B): $0.000987
Naive A + B (double counts A):            $0.001641


## Cell 11: Compaction

A long conversation can approach the context limit. When that happens the SDK **compacts** the conversation automatically: it summarizes the older history to free space and keeps the recent exchanges and key decisions. When it does, the stream contains a `SystemMessage` whose `subtype` is `compact_boundary`.

Compaction frees space, but it **replaces older messages with a summary**. Specific details from early in the conversation may not survive in full. That is the trade you accept.

You do not have to wait for the limit. Sending `/compact` as a prompt triggers compaction on demand. This cell uses that to show the effect safely:

1. Start a `ClaudeSDKClient` session and have the agent read the inventory.
2. Print the context usage.
3. Send `/compact`.
4. Look through the messages that come back for a `SystemMessage` with `subtype == "compact_boundary"`.
5. Print the context usage again.

The cell prints the `subtype` of every `SystemMessage` it sees, so you can see where the compaction marker appears in the stream.

There is also a `PreCompact` hook that lets you run your own code just before compaction, for example to archive the full transcript. We only mention it here and will not use it in this notebook.

Look for `totalTokens` to be **lower** after compaction than before it. The exact drop varies, because the summary has its own size and the fixed parts of the context, such as the system prompt and tool definitions, are not removed.

In [11]:
# Cell 11: compact the conversation and measure the effect
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, ResultMessage, SystemMessage

options = ClaudeAgentOptions(
    allowed_tools=["Read"],
    model=MODEL_NAME,
    cwd="/content",
)

compact_seen = False

async with ClaudeSDKClient(options=options) as client:
    await client.query(
        f"Read {INVENTORY} and tell me which seed has the most packets. "
        "Answer in one sentence."
    )
    async for message in client.receive_response():
        pass
    show_context("Before /compact", await client.get_context_usage())

    print("\nSending /compact ...")
    await client.query("/compact")
    async for message in client.receive_response():
        if isinstance(message, SystemMessage):
            print(f"  SystemMessage subtype: {message.subtype}")
            if message.subtype == "compact_boundary":
                compact_seen = True

    print(f"compact_boundary seen: {compact_seen}")
    show_context("After /compact", await client.get_context_usage())

--- Before /compact ---
  totalTokens: 30,712
  maxTokens:   1,000,000
  percentage:  3.00%
  largest categories:
    Messages: 20,014
    System tools (deferred): 12,216
    System tools: 8,590
    Skills: 1,966

Sending /compact ...
  SystemMessage subtype: status
  SystemMessage subtype: status
  SystemMessage subtype: init
  SystemMessage subtype: compact_boundary
compact_boundary seen: True
--- After /compact ---
  totalTokens: 13,146
  maxTokens:   1,000,000
  percentage:  1.00%
  largest categories:
    System tools (deferred): 12,216
    System tools: 8,525
    Messages: 2,513
    Skills: 1,966


## Cell 12: Guardrails that stop a run

An agent decides for itself how many steps to take. That is powerful, and it is also how a run can get out of hand. Two options put a ceiling on it:

| Option | What it limits | Result subtype when hit |
| --- | --- | --- |
| `max_turns` | How many tool-use round trips the loop may take | `error_max_turns` |
| `max_budget_usd` | The estimated spend at which the loop stops | `error_max_budget_usd` |

When a limit is hit, the run ends with a `ResultMessage` carrying one of those error subtypes. A single-shot `query()` call then **raises an exception** after it has yielded that result. The raise is intentional, which is why the loop must be wrapped in `try/except`. The `run_once` helper from Cell 10 already does that and hands the error text back to you.

Two runs follow:

1. **`max_turns=1`** with a prompt that needs several separate tool calls. One round trip is not enough, so the run should stop early.
2. **`max_budget_usd=0.0001`**, a very small budget. The check happens after a model response arrives, so the response that crosses the cap still completes and counts. Spend can therefore end up above the cap. In practice, leave headroom when you pick a real budget.

For each run the cell prints the `subtype`, the cost and the exception text. Notice that the cost is still reported on an error result. You paid for the work done before the stop.

The lesson is that limits protect you from runaway runs. This notebook does not cover what to do after a limit is hit.

In [12]:
# Cell 12: stop runs with max_turns and max_budget_usd
STEPWISE_PROMPT = (
    f"Do this one step at a time, with one tool call per step. "
    f"Step 1: read {NOTES}. "
    f"Step 2: read {INVENTORY}. "
    f"Step 3: tell me which seed has the most packets and quote the first line of the notes."
)

guardrail_runs = [
    ("max_turns=1", {"max_turns": 1}, "One round trip is too few for a three-step task."),
    ("max_budget_usd=0.0001", {"max_budget_usd": 0.0001}, "The budget is smaller than one real response."),
]

for label, limits, explanation in guardrail_runs:
    result, error = await run_once(STEPWISE_PROMPT, **limits)
    print(f"=== {label} ===")
    print(f"Why it stops: {explanation}")
    if result is None:
        print("No ResultMessage arrived.")
    else:
        print(f"subtype:    {result.subtype}")
        print(f"num_turns:  {result.num_turns}")
        print(f"cost:       {money(result.total_cost_usd)}")
    print(f"exception:  {error}")
    print()

=== max_turns=1 ===
Why it stops: One round trip is too few for a three-step task.
subtype:    error_max_turns
num_turns:  2
cost:       $0.000721
exception:  ResultError: Claude Code returned an error result: Reached maximum number of turns (1) (exit code: 1)

=== max_budget_usd=0.0001 ===
Why it stops: The budget is smaller than one real response.
subtype:    error_max_budget_usd
num_turns:  1
cost:       $0.000139
exception:  ResultError: Claude Code returned an error result: Reached maximum budget ($0.0001) (exit code: 1)



## Cell 13: Effort is a cost lever

The `effort` option controls how much reasoning Claude applies on each response. The supported values in this SDK version are `"low"`, `"medium"`, `"high"`, `"xhigh"` and `"max"`. Lower effort uses fewer tokens per turn, which can reduce cost and speed things up. Not every model supports it, and Claude Haiku 5.5 does.

If you leave `effort` out, Claude Code chooses the level itself. The models page lists `medium` as the default for Claude Haiku 5.5. Effort is separate from the on or off switch for extended thinking.

This cell runs the **same** read-the-inventory prompt twice: once with the default, and once with `effort="low"`. For each run it prints:

- `output_tokens` from the `ResultMessage`, because that is where the real output count lives
- `num_turns`
- the cache fields from `usage`: `cache_creation_input_tokens` (cache write), `cache_read_input_tokens` (cache read) and `input_tokens`
- the cost in dollars, as reported by `total_cost_usd`

Do not expect `low` to be cheaper on every run. Agent output varies, and on a simple task like this one the difference between runs can be larger than the effect of `effort`. A lower effort run can even write a longer answer.

Look at the cache columns before you judge the dollar figure. Cache reads cost roughly one-tenth of ordinary input, and cache writes cost more than ordinary input. The prompt cache expires after a few minutes, so the first call after a pause can pay for a cache write while the next call reads it cheaply. If the two runs show different cache columns, the cost gap comes from caching state, not from `effort`.

For a fair comparison, run the cell twice and read the second result, when the cache is warm for both runs. Then compare output tokens, turns and dollars together, and check that the answer is still correct.

Think of effort as a lever for routine tasks. A simple lookup rarely needs deep reasoning.

In [13]:
# Cell 13: default effort versus effort="low"
runs = [
    ("default", {}),
    ('effort="low"', {"effort": "low"}),
]

print(
    f"{'run':<14} {'output':>8} {'turns':>6} {'cache_write':>12} "
    f"{'cache_read':>11} {'input':>7} {'cost':>12}"
)
answers = {}
for label, extra in runs:
    result, error = await run_once(READ_PROMPT, **extra)
    if result is None:
        print(f"{label:<14} no result: {error}")
        continue
    usage = result.usage or {}
    print(
        f"{label:<14} {usage.get('output_tokens', 0):>8,} {result.num_turns:>6} "
        f"{usage.get('cache_creation_input_tokens', 0):>12,} "
        f"{usage.get('cache_read_input_tokens', 0):>11,} "
        f"{usage.get('input_tokens', 0):>7,} "
        f"{money(result.total_cost_usd):>12}"
    )
    answers[label] = result.result

print("\nAnswers:")
for label, answer in answers.items():
    print(f"  {label}: {answer}")

run              output  turns  cache_write  cache_read   input         cost
default             232      2            0      44,886       4    $0.000695
effort="low"        159      2       19,601      25,285       4    $0.002913

Answers:
  default: SL-0417 (rustic squash) has the most packets at 480, though that is far above every other entry (the next highest is 94), so it may be a data-entry error worth checking.
  effort="low": The seed with the most packets is SL-0417 (rustic squash) with 480 packets, well above the next-highest entry of 94.


## Cell 14: Confirm nothing changed

Every agent call in this notebook used only the `Read` tool, so it was never able to change a file. This final cell proves it by recomputing the SHA-256 hash of both files and comparing each one with the baseline recorded in Cell 5. If a hash differed, something would have written to the file.

Run the cell and look for `unchanged` next to both files. The summary table after the cell collects everything this notebook covered.

In [14]:
# Cell 14: both file hashes must match the Cell 5 baseline
paths = {"seed_inventory.txt": INVENTORY, "notes.txt": NOTES}

all_unchanged = True
for name, path in paths.items():
    now = sha256_of(path)
    same = now == BASELINE_HASHES[name]
    all_unchanged = all_unchanged and same
    print(f"{name}: {'unchanged' if same else 'CHANGED'}")

print("\nAll files unchanged:", all_unchanged)

seed_inventory.txt: unchanged
notes.txt: unchanged

All files unchanged: True


## Summary

| Idea | What to remember | Where you saw it |
| --- | --- | --- |
| Context accumulates | The context window does not reset between turns. Tool outputs such as a large file read can add thousands of tokens in one step | Cell 7 |
| Measuring context | `get_context_usage()` returns `totalTokens`, `maxTokens`, `percentage` and `categories`. Read the token count first and the percentage second | Cell 7 |
| Reading a call's cost | The `ResultMessage` carries `total_cost_usd`, `usage` and `num_turns`. `total_cost_usd` can be `None` and is an estimate, not billing | Cells 8 and 9 |
| Estimating it yourself | Token counts times rates, divided by one million. Close to the SDK figure, but not identical to the last digit | Cell 9 |
| Resumed sessions | A resumed call already includes the earlier spend. Read the latest result and do not add results together | Cell 10 |
| Compaction | Compaction frees space but replaces older messages with a summary. `/compact` triggers it on demand | Cell 11 |
| Guardrails | `max_turns` and `max_budget_usd` stop a run. The result subtype is `error_max_turns` or `error_max_budget_usd`, and `query()` raises afterwards | Cell 12 |
| Effort | `effort` is a cost lever. Compare output tokens, turns and dollars, and do not assume `low` is always cheaper | Cell 13 |

For authoritative billing figures, use the Usage page in the Claude Console at https://platform.claude.com/usage. For current rates, use the pricing page at https://platform.claude.com/docs/en/about-claude/pricing.